# Activation Surgery

Surgical modifications to activations: clamping, scaling, rotation,
projection onto/off subspaces, and targeted replacement.

**References:**
- Turner et al. (2023) "Activation Addition: Steering Language Models Without Optimization"
- Li et al. (2024) "Inference-Time Intervention"

## Why This Matters

Activation surgery provides precise tools for modifying internal activations: clamping, scaling, projecting, rotating, and replacing. These fine-grained interventions enable controlled experiments that go beyond simple ablation to test specific mechanistic hypotheses.

**Key references:**
- [Turner et al. (2023) "Activation Addition"](https://arxiv.org/abs/2308.10248) — Steering model behavior by adding vectors to activations
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from irtk import HookedTransformer, HookedTransformerConfig
from irtk.activation_surgery import (
    clamp_activation,
    scale_activation,
    project_activation,
    rotate_activation,
    targeted_replacement,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

tokens = jnp.array([0, 5, 10, 15, 20, 25, 30, 35, 40, 45])
print(f"Model: {cfg.n_layers} layers, d_model={cfg.d_model}")

In [ ]:
# 1. Clamp activations - restrict values to a range
for bounds in [(-0.5, 0.5), (-0.1, 0.1), (0, None)]:
    result = clamp_activation(model, tokens, "blocks.0.hook_resid_pre", pos=-1,
                             min_val=bounds[0], max_val=bounds[1])
    print(f"Clamp [{bounds[0]}, {bounds[1]}]: logit_diff={result['logit_diff']:.4f}, "
          f"n_clamped={result['n_clamped_values']}")

In [ ]:
# 2. Scale activations - amplify or attenuate
print("Scale factor -> logit diff:")
for factor in [0.0, 0.5, 1.0, 2.0, 5.0]:
    result = scale_activation(model, tokens, "blocks.0.hook_resid_pre", pos=-1, scale_factor=factor)
    print(f"  {factor:.1f}x: logit_diff={result['logit_diff']:.4f}, "
          f"norm {result['original_norm']:.3f} -> {result['scaled_norm']:.3f}")

In [ ]:
# 3. Project onto/off a direction
np.random.seed(42)
direction = np.random.randn(cfg.d_model)
direction /= np.linalg.norm(direction)

onto = project_activation(model, tokens, "blocks.1.hook_resid_post", pos=-1,
                         direction=direction, mode="onto")
off = project_activation(model, tokens, "blocks.1.hook_resid_post", pos=-1,
                        direction=direction, mode="off")
print(f"Project ONTO direction: logit_diff={onto['logit_diff']:.4f}, component={onto['component_magnitude']:.4f}")
print(f"Project OFF direction: logit_diff={off['logit_diff']:.4f}, fraction_removed={off['fraction_removed']:.4f}")

In [ ]:
# 4. Rotate activation between directions
from_dir = np.zeros(cfg.d_model); from_dir[0] = 1.0
to_dir = np.zeros(cfg.d_model); to_dir[1] = 1.0
for strength in [0.0, 0.25, 0.5, 0.75, 1.0]:
    result = rotate_activation(model, tokens, "blocks.0.hook_resid_pre", pos=-1,
                              from_dir=from_dir, to_dir=to_dir, strength=strength)
    print(f"Strength {strength:.2f}: logit_diff={result['logit_diff']:.4f}, "
          f"angle={result['rotation_angle']:.3f} rad, component={result['component_in_from']:.4f}")

In [ ]:
# 5. Targeted replacement
zero_vec = np.zeros(cfg.d_model)
rand_vec = np.random.randn(cfg.d_model) * 0.1

for name, val in [("zero", zero_vec), ("random", rand_vec)]:
    result = targeted_replacement(model, tokens, "blocks.0.hook_resid_pre", pos=-1,
                                 replacement_value=val)
    print(f"Replace with {name}: logit_diff={result['logit_diff']:.4f}, "
          f"displacement={result['displacement']:.4f}")